# APTOS 2019 — Fundus Disease Detection Baseline

**Pipeline**: EfficientNet-B7 / Swin-Base → Ordinal Regression → Quadratic Weighted Kappa  
**Hardware**: 2× T4 GPU (Kaggle) với DDP + Mixed Precision  
**Dataset**: [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)

---
| Cell | Chức năng |
|------|----------|
| 1 | Cài đặt thư viện & import modules |
| 2 | **Config** — chỉnh toàn bộ thông số tại đây |
| 3 | Dataset & DataLoader |
| 4 | Khởi tạo model |
| 5 | Training (DDP + AMP) |
| 6 | Evaluation (held-out 10%) + W&B log eval |
| 7 | Generate submission.csv + W&B log submit |
| 8 | Vẽ figures, zip, artifact đầy đủ + W&B đóng run |

In [ ]:
%%time
# ── Cell 1: Cài đặt thư viện & import ────────────────────────────────────────
# Cài đặt các thư viện cần thiết (bỏ comment nếu chạy lần đầu trên Kaggle)

# !pip install -q timm wandb --upgrade

import os
import sys
import json
import torch
import pandas as pd
import numpy as np

%cd /kaggle/working/
!git clone -b triet-freeze https://github.com/tuan8p/fundus-disease-detection.git
%cd fundus-disease-detection
!git pull origin triet-freeze
!pip install -r requirements.txt


REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.dataset   import get_dataloaders
from src.models    import build_model, predict_labels
from src.train     import run_training
from src.evaluate  import run_evaluation
from src.visualize import plot_all
from src.utils     import (
    load_checkpoint,
    generate_submission,
    zip_outputs,
    load_history,
    resume_wandb_run,
    log_eval_phase_to_wandb,
    log_submission_phase_to_wandb,
    log_visualization_phase_to_wandb,
    wandb_finish_pipeline,
    start_pipeline_console_capture,
    stop_pipeline_console_capture,
)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
%%time
# ── Cell 2: CONFIG — Chỉnh toàn bộ thông số tại đây ────────────────────────────

#  Thông số cuối cùng — tối ưu từ 27-case HPO (9 shard × 3 case, 15 epoch/case)
#   Best HPO case: sw5_lr1e4_blr015 có val QWK = 0.9435

# ──────────────────────────────────────────────────────────────────────────────
#  MODEL
# ──────────────────────────────────────────────────────────────────────────────
MODEL_TYPE  = "swinv2_base_384"   # "efficientnet_b7" | "swinv2_base_384"
IMAGE_SIZE  = 384                 # 456 → efficientnet_b7 | 384 → swinv2_base_384

# ──────────────────────────────────────────────────────────────────────────────
#  TRAINING — Hyperparameters 
# ──────────────────────────────────────────────────────────────────────────────
BATCH_SIZE  = 8                   
GRAD_ACCUM_STEPS = 1             
EPOCHS      = 25                  
USE_AMP     = True                
NUM_WORKERS = 4                  

# ── Optimizer (AdamW + Differential LR) ──────────────────────────────────────

LR                 = 1e-4    
BACKBONE_LR_SCALE  = 0.15   
WEIGHT_DECAY       = 5e-5   
ADAMW_BETA1        = 0.9    
ADAMW_BETA2        = 0.999  
ADAMW_EPS          = 1e-8

# ── Freeze strategy ──────────────────────────────────────────────────────────
#  "none"      : Train toàn bộ network (mặc định — HPO dùng none)
#  "head_only" : Chỉ train classifier head (Linear Probing)
#  "partial"   : Unfreeze block/stage cuối + head
FREEZE_STRATEGY = "none"

# ── Scheduler ────────────────────────────────────────────────────────────────

LR_FACTOR      = 0.5              
LR_PATIENCE    = 3                
WARMUP_EPOCHS  = 2                
T_0            = 11               

# ── Augmentation ─────────────────────────────────────────────────────────────
AUGMENT_VERSION = "v2"        

# ── Loss Function ─────────────────────────────────────────────────────────────
import numpy as np
_class_counts = [1805, 370, 999, 193, 295]  
_n_samples    = sum(_class_counts)
_n_classes    = len(_class_counts)
CLASS_WEIGHTS = [round(_n_samples / (_n_classes * c), 4) for c in _class_counts]
print(f"CLASS_WEIGHTS (auto) = {CLASS_WEIGHTS}")

# ──────────────────────────────────────────────────────────────────────────────
#  DATA SPLIT  
# ──────────────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.7                 # 70% train
VAL_RATIO   = 0.2                 # 20% val
TEST_RATIO  = 0.1                 # 10% test 
SEED        = 42

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  (Kaggle environment)
# ──────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/competitions/aptos2019-blindness-detection"
OUTPUT_DIR = "/kaggle/working/outputs"

# ──────────────────────────────────────────────────────────────────────────────
#  WEIGHTS & BIASES
# ──────────────────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "fundus-baseline"
WANDB_RUN_NAME = f"{MODEL_TYPE}_lr{LR:.0e}_blr{BACKBONE_LR_SCALE}_ep{EPOCHS}_aug{AUGMENT_VERSION}"

import wandb
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = api_key
    
    wandb.login(key=api_key)
    print(" Đã login W&B thành công từ Kaggle Secrets!")
    
except Exception as e:
    print(f"Lỗi lấy Kaggle Secrets: {e}")
    
    # Tắt wandb (chuyển sang offline) để notebook KHÔNG BỊ TREO khi chạy Save and Run All
    os.environ["WANDB_MODE"] = "offline" 
    print("Đã chuyển W&B sang chế độ OFFLINE")

# ──────────────────────────────────────────────────────────────────────────────
#  Tạo thư mục output
# ──────────────────────────────────────────────────────────────────────────────
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"),     exist_ok=True)

# Đóng gói toàn bộ config vào dict để truyền vào các hàm src/
CFG = {
    # ── Model ──────────────────────────────────────────────────────────────
    "MODEL_TYPE":          MODEL_TYPE,
    "IMAGE_SIZE":          IMAGE_SIZE,
    # ── Training ───────────────────────────────────────────────────────────
    "BATCH_SIZE":          BATCH_SIZE,
    "GRAD_ACCUM_STEPS":    GRAD_ACCUM_STEPS,
    "EPOCHS":              EPOCHS,
    "USE_AMP":             USE_AMP,
    "NUM_WORKERS":         NUM_WORKERS,
    # ── Optimizer ──────────────────────────────────────────────────────────
    "LR":                  LR,
    "BACKBONE_LR_SCALE":   BACKBONE_LR_SCALE,   
    "WEIGHT_DECAY":        WEIGHT_DECAY,         
    "ADAMW_BETA1":         ADAMW_BETA1,         
    "ADAMW_BETA2":         ADAMW_BETA2,         
    "ADAMW_EPS":           ADAMW_EPS,            
    # ── Freeze & Loss ──────────────────────────────────────────────────────
    "FREEZE_STRATEGY":     FREEZE_STRATEGY,
    "CLASS_WEIGHTS":       CLASS_WEIGHTS,
    # ── Augmentation ───────────────────────────────────────────────────────
    "AUGMENT_VERSION":     AUGMENT_VERSION,      
    # ── Scheduler ──────────────────────────────────────────────────────────
    "LR_FACTOR":           LR_FACTOR,
    "LR_PATIENCE":         LR_PATIENCE,
    "WARMUP_EPOCHS":       WARMUP_EPOCHS,
    "T_0":                 T_0,
    # ── Data & Paths ───────────────────────────────────────────────────────
    "TRAIN_RATIO":         TRAIN_RATIO,
    "VAL_RATIO":           VAL_RATIO,
    "SEED":                SEED,
    "DATA_DIR":            DATA_DIR,
    "OUTPUT_DIR":          OUTPUT_DIR,
    "REPO_ROOT":           REPO_ROOT,            
    # ── W&B ────────────────────────────────────────────────────────────────
    "WANDB_PROJECT":       WANDB_PROJECT,
    "WANDB_RUN_NAME":      WANDB_RUN_NAME,
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k:<22} = {v}")


PIPELINE_LOG_PATH = start_pipeline_console_capture(OUTPUT_DIR)
print(f"\nPipeline console log (toàn bộ print) → {PIPELINE_LOG_PATH}")


In [ ]:
%%time
# ── Cell 3: Dataset & DataLoader ─────────────────────────────────────────────
(
    train_loader,
    val_loader,
    internal_test_loader,
    submit_loader,
    _,
) = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,   
    preprocessing_strategy = "none", 
    aug_version= DATA_AUG
)

# Lấy danh sách image_ids của tập submit
test_csv   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
SUBMIT_IDS = test_csv["id_code"].tolist()

print(f"Train batches        : {len(train_loader):>5}  (~{len(train_loader.dataset)} ảnh)")
print(f"Val batches          : {len(val_loader):>5}  (~{len(val_loader.dataset)} ảnh)")
print(f"Internal test batches: {len(internal_test_loader):>5}  (~{len(internal_test_loader.dataset)} ảnh)")
print(f"Submit batches       : {len(submit_loader):>5}  (~{len(submit_loader.dataset)} ảnh)")

# Kiểm tra shape batch đầu tiên
sample_imgs, sample_labels = next(iter(train_loader))
print(f"\nSample batch — images: {sample_imgs.shape}, labels: {sample_labels.shape}")
print(f"Label range: {sample_labels.min().item():.0f} – {sample_labels.max().item():.0f}")

In [ ]:
%%time
# ── Cell 4: Khởi tạo Model ────────────────────────────────────────────────────
# Tải pretrained ImageNet weights từ HuggingFace Hub qua timm.
# Classifier head được thay bằng 1 neuron cho ordinal regression.

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = build_model(
    MODEL_TYPE,
    pretrained=True,
    freeze_strategy=FREEZE_STRATEGY,  
).to(device)


total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model type       : {MODEL_TYPE}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Device           : {device}")

# Kiểm tra forward pass với 1 batch nhỏ
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    out   = model(dummy)
    print(f"Forward pass OK  : input {tuple(dummy.shape)} → output {tuple(out.shape)}")

In [ ]:
%%time
# ── Cell 5: Training ─────────────────────────────────────────────────────────
run_training(CFG)

In [ ]:
%%time
# ── Cell 6: Evaluation trên tập test đã chia ─────────────────

import torch.nn as nn

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Tạo lại model và load best checkpoint
eval_model = build_model(MODEL_TYPE, pretrained=False).to(device)
ckpt_path  = os.path.join(OUTPUT_DIR, "checkpoints", f"best_model_{MODEL_TYPE}.pth")
ckpt_meta  = load_checkpoint(eval_model, ckpt_path, device)

criterion = nn.SmoothL1Loss()

# Tạo lại internal_test_loader 
_, _, eval_test_loader, _, _ = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,
)

# Chạy evaluation
EVAL_METRICS = run_evaluation(
    model      = eval_model,
    loader     = eval_test_loader,
    criterion  = criterion,
    device     = device,
    output_dir = OUTPUT_DIR,
)

EVAL_TXT = os.path.join(OUTPUT_DIR, "evaluation_metrics.txt")
WANDB_RUN = resume_wandb_run(OUTPUT_DIR, CFG)
log_eval_phase_to_wandb(WANDB_RUN, EVAL_METRICS, EVAL_TXT)

In [ ]:
%%time
# ── Cell 7: Tạo Submission CSV ──────────────────────────────────────────


# Tạo lại submit_loader 
_, _, _, sub_loader, _ = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,
)

sub_path = generate_submission(
    model        = eval_model,
    submit_loader= sub_loader,
    image_ids    = SUBMIT_IDS,
    output_dir   = OUTPUT_DIR,
    device       = device,
)

# Preview submission
sub_df = pd.read_csv(sub_path)
print(f"\nSubmission preview ({len(sub_df)} rows):")
print(sub_df.head(10).to_string(index=False))
print(f"\nDistribution:\n{sub_df['diagnosis'].value_counts().sort_index().to_string()}")

# W&B: phân phối submission + artifact CSV (nếu Cell 6 chưa chạy thì resume run)
try:
    WANDB_RUN
except NameError:
    WANDB_RUN = resume_wandb_run(OUTPUT_DIR, CFG)
log_submission_phase_to_wandb(WANDB_RUN, sub_path, sub_df)

In [ ]:
%%time
# ── Cell 8: Vẽ Figures & Zip Outputs ─────────────────────────────────────────

import matplotlib
matplotlib.use('Agg')   
import matplotlib.pyplot as plt

# Đọc history từ file JSON đã lưu trong quá trình training
HISTORY = load_history(OUTPUT_DIR)

# Vẽ và lưu tất cả figure
saved_figures = plot_all(
    metrics    = EVAL_METRICS,
    history    = HISTORY,
    output_dir = OUTPUT_DIR,
    model_type = MODEL_TYPE,
)

# Hiển thị figure trong notebook
import matplotlib.image as mpimg
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = ["Confusion Matrix", "Training Curves", "Per-class Recall"]
for ax, path, title in zip(axes, saved_figures, titles):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Zip toàn bộ outputs
zip_path = zip_outputs(OUTPUT_DIR)

# Tóm tắt cuối)
print("\n" + "="*50)
print("PIPELINE HOÀN TẤT")
print("="*50)
print(f"  Best model  : {os.path.join(OUTPUT_DIR, 'checkpoints', f'best_model_{MODEL_TYPE}.pth')}")
print(f"  Submission  : {os.path.join(OUTPUT_DIR, 'submission.csv')}")
print(f"  Report      : {os.path.join(OUTPUT_DIR, 'classification_report.txt')}")
print(f"  Console log : {os.path.join(OUTPUT_DIR, 'pipeline_full_console.log')}")
print(f"  Figures     : {os.path.join(OUTPUT_DIR, 'figures/')}")
print(f"  Archive     : {zip_path}")
print(f"\n  QWK (held-out test) : {EVAL_METRICS['qwk']:.4f}")
print(f"  Accuracy            : {EVAL_METRICS['accuracy']:.4f}")
print(f"  Macro F1            : {EVAL_METRICS['macro_f1']:.4f}")
print(f"  Balanced Accuracy   : {EVAL_METRICS['balanced_accuracy']:.4f}")

# Đóng tee trước khi upload lên W&B 
stop_pipeline_console_capture()

try:
    WANDB_RUN
except NameError:
    WANDB_RUN = resume_wandb_run(OUTPUT_DIR, CFG)
log_visualization_phase_to_wandb(WANDB_RUN, saved_figures, OUTPUT_DIR, zip_path)
wandb_finish_pipeline(WANDB_RUN)